# 01 · 原始数据检查（M2）

检查原始事件 CSV 的结构、事件类型、缺失与日期覆盖。为了在普通笔记本上快速复现，这里用 **Cosmetics Shop 2019-Oct**（约 85 MB 压缩）做演示；主数据集 Multi-category Store 的每月文件更大（数 GiB），检查口径完全一致，正式数字见 `reports/tables/aggregation_check.csv` 与 `reports/aggregation_check_report.md`。

## 环境

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (cand / 'src' / 'demandlens').exists():
        ROOT = cand
        break
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd

pd.show_versions = None  # 保持输出干净
print('pandas', pd.__version__)

pandas 2.2.3


## 获取演示用原始文件

公开数据集可用 kagglehub 匿名下载；已缓存时直接复用（需求 D5：换月份只改配置，不改代码）。

In [2]:
import kagglehub

RAW = kagglehub.dataset_download(
    'mkechinov/ecommerce-events-history-in-cosmetics-shop',
    path='2019-Oct.csv')
print('文件:', RAW)

  0%|          | 0.00/83.1M [00:00<?, ?B/s]

  1%|          | 1.00M/83.1M [00:01<01:33, 918kB/s]

  2%|▏         | 2.00M/83.1M [00:01<00:49, 1.73MB/s]

  5%|▍         | 4.00M/83.1M [00:01<00:21, 3.82MB/s]

  6%|▌         | 5.00M/83.1M [00:01<00:18, 4.36MB/s]

  8%|▊         | 7.00M/83.1M [00:01<00:12, 6.41MB/s]

... (47 outputs left out)

 99%|█████████▊| 82.0M/83.1M [00:11<00:00, 7.24MB/s]

100%|██████████| 83.1M/83.1M [00:11<00:00, 7.80MB/s]

Extracting zip of 2019-Oct.csv...


文件: /Users/sonderism/.cache/kagglehub/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop/versions/6/2019-Oct.csv


## 结构：列、类型、前几行

In [3]:
head = pd.read_csv(RAW, nrows=5)
head

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,cart,5773203,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03 UTC,cart,5773353,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07 UTC,cart,5881589,2151191071051219817,NaN,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07 UTC,cart,5723490,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15 UTC,cart,5881449,1487580013522845895,NaN,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


In [4]:
sample = pd.read_csv(RAW, usecols=['event_time', 'event_type',
                                   'category_id', 'category_code'],
                     dtype='string', nrows=2_000_000)
sample.dtypes.to_dict()

{'event_time': string[python],
 'event_type': string[python],
 'category_id': string[python],
 'category_code': string[python]}

## 事件类型构成

注意：该数据集含 `remove_from_cart`，它**不属于**契约三类（view/cart/purchase），聚合时会被排除并单独计数。

In [5]:
sample['event_type'].str.strip().str.lower().value_counts()

event_type
view                882690
cart                648292
remove_from_cart    356340
purchase            112678
Name: count, dtype: Int64

## 缺失与日期覆盖

In [6]:
date_str = sample['event_time'].str.slice(0, 10)
valid_date = pd.to_datetime(date_str, format='%Y-%m-%d', errors='coerce')

print('category_code 缺失比例: %.1f%%' % (sample['category_code'].isna().mean() * 100))
print('非法日期行数:', int(valid_date.isna().sum()))
print('日期范围:', valid_date.min(), '..', valid_date.max())
print('类别数(前 200 万行):', sample['category_id'].nunique())

category_code 缺失比例: 98.4%
非法日期行数: 0
日期范围: 2019-10-01 00:00:00 .. 2019-10-15 00:00:00
类别数(前 200 万行): 482


## 与契约的对照

| 检查点 | 结论 |
|---|---|
| 列结构 | 与 Multi-category Store 同构，读取器可复用 |
| 事件类型 | 含契约外类型，须过滤（aggregate.py 已处理） |
| category_id | 可解析为整数，可作为聚合主键 |
| category_code | 大量缺失，不能作为主键；仅作可读标签 |

更完整的多类别商店十月检查（4200 万事件）见小组 11 月 5 日数据集交流记录；本流水线逐月的正式核对数字见 `reports/tables/aggregation_check.csv`。